In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))
import timeit

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
import numpy as np
from gridr.core.convolution.fft_filtering import fft_array_filter
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE = True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook  # enables plot interface in J notebook
    output_notebook()


def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.

    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing:
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape

    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)

    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)

    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x) ** 2 + (y_grid / sigma_y) ** 2))

    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum

    return x, y, kernel


def _add_image_axes(fig, img, left, bottom, fig_w, fig_h, scale, **imshow_kw):
    """Add an axes displaying img at native resolution, positioned in screen pixels."""
    h, w = img.shape[:2]
    ax = fig.add_axes([left / fig_w, bottom / fig_h,
                       w * scale / fig_w, h * scale / fig_h])
    ax.imshow(img, cmap="gray", interpolation="none", resample=False,
              aspect="equal", **imshow_kw)
    # imshow puts pixel centers on integers, so pixel edges sit at +/- 0.5
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    ax.set_axis_off()
    return ax


def _window_rect(window, **kwargs):
    """Build a matplotlib Rectangle from an inclusive window ((r0, r1), (c0, c1))."""
    (r0, r1), (c0, c1) = window
    return Rectangle((c0 - 0.5, r0 - 0.5), c1 - c0 + 1, r1 - r0 + 1,
                     fill=False, **kwargs)


def valid_window_from_prod(window_prod, kernel_shape):
    """Compute the valid window (inclusive, full-image frame) from the same window.

    The kernel is assumed to be padded to an odd size: K_odd = K + (1 - K % 2).
    Returns None if the valid region is empty along any axis (N < K_odd).
    """
    window_prod = np.asarray(window_prod)
    k = np.asarray(kernel_shape[:2])
    radius = (k + (1 - k % 2)) // 2
    window_valid = window_prod + np.stack([radius, -radius], axis=1)
    if np.any(window_valid[:, 1] < window_valid[:, 0]):
        return None
    return window_valid


def _crop(image, window):
    """Extract the region of image covered by an inclusive window."""
    (r0, r1), (c0, c1) = window
    return image[r0:r1 + 1, c0:c1 + 1]


@mpl_plot_wrapper
def plot_image_output_mode(image_full, window_prod, kernel_shape, prefix=None,
                           scale=1, dpi=100):
    """Plot the full output with its same/valid regions, then each region alone.

    Each image pixel is rendered as exactly scale x scale screen pixels
    (scale must be an integer), with no interpolation. Panels are vertically
    centered on each other and linked by arrows.
    """
    window_prod = np.asarray(window_prod)
    window_valid = valid_window_from_prod(window_prod, kernel_shape)
    h, w = image_full.shape[:2]

    # Panels: (image, window in full coordinates or None, rect style)
    panels = [(image_full, None, None),
              (_crop(image_full, window_prod), window_prod,
               dict(edgecolor="tab:blue", linestyle="-", label="same"))]
    if window_valid is not None:
        window_valid = np.asarray(window_valid)
        panels.append((_crop(image_full, window_valid), window_valid,
                       dict(edgecolor="tab:green", linestyle="--", label="valid")))

    # Layout in screen pixels (integers to stay on the pixel grid)
    m_left, m_right, m_top, m_bottom = 10, 10, 24, 30
    gap = 40  # horizontal room for the arrows
    ax_h = h * scale  # the full image is the tallest panel
    widths = [p[0].shape[1] * scale for p in panels]
    fig_w = m_left + sum(widths) + gap * (len(panels) - 1) + m_right
    fig_h = ax_h + m_top + m_bottom

    fig = plt.figure(figsize=(fig_w / dpi, fig_h / dpi), dpi=dpi)
    norm_kw = dict(vmin=np.nanmin(image_full), vmax=np.nanmax(image_full))

    left = m_left
    y_center = m_bottom + ax_h / 2
    handles = []
    for i, (img, window, style) in enumerate(panels):
        p_h = img.shape[0] * scale
        bottom = m_bottom + (ax_h - p_h) // 2
        ax = _add_image_axes(fig, img, left, bottom, fig_w, fig_h, scale, **norm_kw)

        if i == 0:
            # Full panel: overlay every region
            handles.append(_window_rect([[0, h - 1], [0, w - 1]],
                                        edgecolor="tab:red", linewidth=1,
                                        linestyle="-", label="full"))
            handles += [_window_rect(win, linewidth=1, **st)
                        for _, win, st in panels[1:]]
            for patch in handles:
                ax.add_patch(patch)
        else:
            # Cropped panel: frame it with its region color
            ph, pw = img.shape[:2]
            ax.add_patch(_window_rect([[0, ph - 1], [0, pw - 1]],
                                      linewidth=1, **style))
            # Arrow from the previous panel
            x0 = left - gap + 6
            x1 = left - 6
            fig.add_artist(FancyArrowPatch(
                (x0 / fig_w, y_center / fig_h), (x1 / fig_w, y_center / fig_h),
                transform=fig.transFigure, arrowstyle="-|>",
                mutation_scale=10, linewidth=1, color="0.3"))

        left += widths[i] + gap

    win_h, win_w = window_prod[:, 1] - window_prod[:, 0] + 1
    fig.text(0.5, 1 - 4 / fig_h,
             f"full ({h}x{w}) - input ({win_h}x{win_w}) - "
             f"kernel ({kernel_shape[0]}x{kernel_shape[1]})",
             ha="center", va="top", fontsize=8)
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0),
               ncol=len(handles), frameon=False, fontsize=7)
    return fig


def benchmark(func, *, repeat=7, number=None, warmup=1):
    """Measure the mean execution time of ``func``, in seconds.

    The first calls are discarded: they include the FFT planning done by the
    backend, which is cached afterwards and would bias the measurement.

    :param func: callable taking no argument
    :param repeat: number of timing runs the statistics are computed from
    :param number: calls per run (auto-calibrated when None)
    :param warmup: number of discarded calls before timing
    :return: a tuple (mean, standard deviation, min) in seconds
    """
    for _ in range(warmup):
        func()
    timer = timeit.Timer(func)
    if number is None:
        number, _ = timer.autorange()
    times = np.array(timer.repeat(repeat=repeat, number=number)) / number
    return times.mean(), times.std(), times.min()


mandrill_0 = mandrill[0]

# The `gaussian_kernel_2d` function implements the formula given in the first tutorial.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1), sigma=gaussian_sigma)

# Limiting computation to a window

This third tutorial describes how to restrict the computation to a rectangular region of
a larger input array, and what distinguishes this from simply cropping the image.

**What you'll learn**

- How the ``win`` argument defines the production window inside the input array
- Why passing ``win`` is not equivalent to passing a cropped view, and where the boundary condition still applies
- That the result is identical to filtering the whole image and cropping it afterwards
- How much computation time the production window saves

## The production window

`fft_array_filter` accepts a `win` parameter that restricts the computation to a
rectangular region of the input array, called the production window.

It is not the same as passing a cropped view as input. A crop discards the surrounding
pixels, which then have to be recreated by the boundary condition, and the values near
the edges of the region are affected accordingly. With `win`, those pixels are still
available in the input array and are used as they are: the result is identical to
filtering the whole image and cropping it, and the boundary condition only applies where
the production window touches the actual image borders.

It also differs from filtering the whole image and cropping the result afterwards, this
time in terms of computation cost: only the requested region, extended by the margin the
kernel needs, is convolved, so the cost scales with the size of the window rather than
with the size of the image.

Both points are illustrated below.

### Filtering a cropped view

We first filter a 192x192 crop of the mandrill image, passed as a standalone array. The
surrounding pixels are no longer part of the input, so the boundary condition applies on
all four sides.

In [ ]:
# Crop the input
filtered_result_none_full = fft_array_filter(
        arr=mandrill_0[0:192, 160:352],
        kernel=kernel,
        win=None,
        boundary="none",
        out_mode="full",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
    )

In [ ]:
_ = plot_image_output_mode(filtered_result_none_full.data,
                           filtered_result_none_full.window, kernel.shape,
                           prefix="003_cropped_view_filtering")

With `boundary="none"` the extrapolated values are zeros, and the darkening is visible
along the whole perimeter of the same window.

### Filtering with a production window

The same region is now requested through `win`, with the whole image passed as input.

In [ ]:
# Provide a window
filtered_result_none_full_win = fft_array_filter(
        arr=mandrill_0,
        kernel=kernel,
        win=[[0, 191], [160, 351]],
        boundary="none",
        out_mode="full",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
    )

In [ ]:
_ = plot_image_output_mode(filtered_result_none_full_win.data,
                           filtered_result_none_full_win.window, kernel.shape,
                           prefix="003_window_filtering")

With a production window, the margins are read from the surrounding pixels of the full
image, and the three edges that lie inside the image show no darkening. The top edge is
the exception: the window starts at row 0, which is also the first row of the image, so
no data is available above it and the boundary condition applies there. With
`boundary="none"` the missing values are zeros, hence the darker band.

## Computation cost

Restricting the computation with `win` returns the same values as filtering the whole
image and cropping the result afterwards, but not at the same cost. The two are compared
below.

In [ ]:
def with_win():
    return fft_array_filter(
        arr=mandrill_0,
        kernel=kernel,
        win=[[0, 191], [160, 351]],
        boundary="none",
        out_mode="same",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
    ).data


def without_win():
    return fft_array_filter(
        arr=mandrill_0,
        kernel=kernel,
        win=None,
        boundary="none",
        out_mode="same",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
    ).data[0:192, 160:352]

Both approaches return the same values:

In [ ]:
np.testing.assert_allclose(with_win(), without_win())

On our reference machine, restricting the computation with `win` is about ten times
faster than filtering the whole image and cropping the result (1.6 ms versus 16 ms).
These figures are quoted rather than computed when the documentation is built, since
they depend on the machine and on the runtime conditions. Uncomment the cell below to
measure the ratio on your own hardware.

In [ ]:
# get mean, std and min time in seconds
# benchmark(with_win, repeat=10, number=10, warmup=5)
# benchmark(without_win, repeat=10, number=10, warmup=5)

## What's next

The output has kept the resolution of the input so far. The next tutorial, *Zooming out
and aliasing*, shows how to resample the output with the `zoom` argument, and why the
kernel has to be chosen together with the decimation factor.